# Module 3: Prefill, Decode, and the KV Cache

In Module 1 you decided to run your own server. Now you open the box. This module is not a tour of metrics. You build the mechanism yourself, derive the memory from first principles, and then measure both on the GPU you own.

By the end you will have implemented attention by hand, written the naive generation loop and watched it cost `O(n^2)`, added a KV cache and watched it drop to `O(n)`, derived exactly how many bytes that cache costs per token, and recovered your card's memory bandwidth from nothing but the speed it generates tokens. Every number here you will have either derived or measured. None of it is asserted.

## Learning objectives
- Implement single-head attention and see why generating one token needs the keys and values of every token before it
- Write naive autoregressive generation and measure its `O(n^2)` cost, the `np + n(n-1)/2` the math predicts
- Add a KV cache, measure the drop to `O(n)`, and name the exact trade you made
- Derive `KV_bytes_per_token = 2 * layers * kv_heads * head_dim * dtype_bytes` and compute it for your model
- Separate prefill (a GEMM, compute-bound) from decode (a GEMV, memory-bound) using arithmetic intensity, not intuition
- Recover your GPU's effective memory bandwidth from the measured decode rate, and show batching lifts decode off that floor
- Reuse the KV cache across requests with prefix caching, and watch an agent's resent prompt make prefill free

## Prerequisites
- Finished Modules 1 and 2, with a working `VLLM_HOST` and resolved settings
- The mechanism sections (1 to 3, 6) are pure `numpy` and run anywhere, no GPU needed
- The measurement sections (4, 7, 8) call your live vLLM endpoint
- About 25 minutes

References: [The Physics of LLM Inference, Ch.2 to 3](https://docs.vllm.ai) &middot; [PagedAttention paper](https://arxiv.org/abs/2309.06180) &middot; [vLLM production metrics](https://docs.vllm.ai/en/latest/design/metrics/) &middot; [Anatomy of vLLM](https://blog.vllm.ai/2025/09/05/anatomy-of-vllm.html)

## 1. Setup

Two dependencies do everything here: `numpy` to build the mechanism, and the OpenAI client to measure the real server. We reinstall so the notebook stands alone.

In [ ]:
%pip install -q numpy matplotlib "openai>=1.40" requests

In [ ]:
# Resolve your endpoint, and seed a deterministic RNG for the by-hand sections.
import os, sys, time, math
import numpy as np
sys.path.insert(0, os.path.abspath(".."))
from common.config import get_settings, build_client
from common import metrics

settings = get_settings()
rng = np.random.default_rng(0)
print("model   :", settings.model_name)
print("endpoint:", settings.vllm_host)

**What you should see:** your model id and endpoint. The mechanism you build next is the same one this server runs, just at a size you can read.

## 2. One forward pass: attention, by hand

A transformer layer does two things to a sequence of token vectors: it mixes them with attention, then transforms each one through a feed-forward network. Attention is the part that makes generation expensive, so build it.

Each token is projected three ways: a **query** (what am I looking for), a **key** (what do I offer), and a **value** (what I pass on if attended to). A token's output is a weighted sum of all values, weighted by how much its query matches each key:

`scores = Q @ K^T / sqrt(d)`, then `softmax`, then `@ V`.

The shape of `scores` is the whole story. In prefill you hold `p` prompt tokens, so `Q` and `K` are both `(p, d)` and `scores` is `(p, p)`: a matrix times a matrix, a **GEMM**. Keep an eye on that shape; it changes in decode and that change is the reason decode is slow.

In [ ]:
# Single-head attention. Q,K,V are token vectors projected three ways.
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, causal=False):
    d = Q.shape[-1]
    scores = (Q @ K.T) / math.sqrt(d)          # (q_len, kv_len): the attention matrix
    if causal:
        ql, kl = scores.shape
        scores[np.triu(np.ones((ql, kl), bool), k=kl - ql + 1)] = -np.inf
    return softmax(scores, axis=-1) @ V        # (q_len, d)

# Prefill: a 10-token prompt at model dim 64. All tokens at once.
d, p = 64, 10
X = rng.standard_normal((p, d))                 # token embeddings
Wq, Wk, Wv = (rng.standard_normal((d, d)) for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv
out = attention(Q, K, V, causal=True)
print("scores (the attention matrix):", (Q @ K.T).shape, " <- p x p, a GEMM")
print("layer output                 :", out.shape)

**What you should see:** a `(10, 10)` score matrix and a `(10, 64)` output. The score matrix is `p x p`: every token scored against every token, in one parallel matrix multiply. That is prefill. The GPU loves this shape because it does a lot of compute per byte it reads. Section 6 makes that precise.

## 3. Generate one token: the naive way, and why it is `O(n^2)`

A language model writes one token at a time. It samples a token, appends it, and runs again. The naive loop reruns the whole model on the whole sequence every step.

That is correct and wasteful. When you generate token 100, the model reprocesses tokens 0 to 99, recomputing attention that has not changed since you generated token 99. Count the work. Generating `n` tokens after a `p`-token prompt:

- token 1 is a forward pass over `p` tokens
- token 2 over `p+1`
- token `n` over `p+n-1`

Total token-forward-passes = `sum(p+i for i in 0..n-1)` = `np + n(n-1)/2`. That is `O(n^2)`. Build it and watch the clock bend.

In [ ]:
# A tiny 6-layer pre-norm transformer so the work is real. Weights are scaled by
# 1/sqrt(fan_in) and each sublayer is RMSNorm'd, exactly like a real model, so the
# residual stream stays stable across layers. Same structure, readable size.
L = 6
sc = lambda a, b: rng.standard_normal((a, b)) / math.sqrt(a)
Wqs=[sc(d,d) for _ in range(L)]; Wks=[sc(d,d) for _ in range(L)]
Wvs=[sc(d,d) for _ in range(L)]; Wos=[sc(d,d) for _ in range(L)]
Wf1=[sc(d,4*d) for _ in range(L)]; Wf2=[sc(4*d,d) for _ in range(L)]

def rms_norm(x):
    return x / np.sqrt((x**2).mean(-1, keepdims=True) + 1e-6)

def layer_full(x, i):                    # process ALL tokens through layer i
    h = rms_norm(x)
    x = x + attention(h@Wqs[i], h@Wks[i], h@Wvs[i], causal=True) @ Wos[i]
    h = rms_norm(x)
    return x + np.maximum(0, h@Wf1[i]) @ Wf2[i]     # ReLU feed-forward

def naive_generate(prompt, n_new):
    seq = prompt.copy()
    for _ in range(n_new):
        x = seq
        for i in range(L):
            x = layer_full(x, i)         # the whole sequence, every step
        seq = np.vstack([seq, x[-1:]])   # append the new token's vector
    return seq

prompt = rng.standard_normal((8, d))
ns = [16, 32, 64, 96, 128]
naive_ms = []
for n in ns:
    t = time.perf_counter(); naive_generate(prompt, n); naive_ms.append((time.perf_counter()-t)*1000)
    print(f"naive  n={n:>3}: {naive_ms[-1]:7.1f} ms   (token-passes: {8*n + n*(n-1)//2:>6})")

**What you should see:** the time per run growing faster than the token count. Doubling `n` from 64 to 128 more than doubles the time. The `token-passes` column is the `np + n(n-1)/2` from the derivation, and the wall clock tracks it. This is the cost you pay for recomputing the past on every step.

## 4. The KV cache: store K and V, recompute nothing

Here is the insight the whole field is built on. A token's key and value depend only on that token, never on what comes after. So once you compute them, they never change. Store them.

Now decode is cheap. For each new token you compute its query, key, and value once, append the key and value to a cache, and attend the single new query against the full cached keys. The expensive projections run on **one** token, not the whole sequence. Watch the same generation go linear.

In [ ]:
# Cached generation: K,V for the new token only, appended to a per-layer cache.
def cached_generate(prompt, n_new):
    kc=[None]*L; vc=[None]*L
    x = prompt.copy()                                  # prefill fills the cache once
    for i in range(L):
        h = rms_norm(x); k, v = h@Wks[i], h@Wvs[i]; kc[i], vc[i] = k, v
        x = x + attention(h@Wqs[i], k, v, causal=True) @ Wos[i]
        h = rms_norm(x); x = x + np.maximum(0, h@Wf1[i]) @ Wf2[i]
    tok = x[-1:]
    for _ in range(n_new):
        x = tok                                        # ONE token, not the sequence
        for i in range(L):
            h = rms_norm(x); k, v = h@Wks[i], h@Wvs[i]
            kc[i] = np.vstack([kc[i], k]); vc[i] = np.vstack([vc[i], v])
            x = x + attention(h@Wqs[i], kc[i], vc[i]) @ Wos[i]   # (1, s): a GEMV
            h = rms_norm(x); x = x + np.maximum(0, h@Wf1[i]) @ Wf2[i]
        tok = x
    return kc

cached_ms = []
for n in ns:
    t = time.perf_counter(); cached_generate(prompt, n); cached_ms.append((time.perf_counter()-t)*1000)
    print(f"cached n={n:>3}: {cached_ms[-1]:7.1f} ms")

In [ ]:
# Plot the two side by side: the quadratic you pay for, the linear you keep.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(ns, naive_ms, "o-", label="naive: reprocess the sequence  (O(n^2))")
ax.plot(ns, cached_ms, "o-", label="cached: process one token     (O(n))")
ax.set_xlabel("tokens generated (n)"); ax.set_ylabel("time (ms)")
ax.set_title("The KV cache turns generation from quadratic to linear")
ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout()
print(f"at n={ns[-1]}: naive {naive_ms[-1]:.0f} ms vs cached {cached_ms[-1]:.0f} ms "
      f"({naive_ms[-1]/cached_ms[-1]:.1f}x)")

**What you should see:** the naive curve bending upward while the cached curve stays close to a straight line, and a speedup that grows with `n`. You just traded `O(n^2)` redundant compute for `O(n)` memory: you keep every key and value instead of recomputing them. That memory is the KV cache, and the rest of this module is about how much it costs and what it does to the GPU.

## 5. Sizing the cache, derived then measured

The cache holds, for every layer and every token, one key vector and one value vector. Count the bytes:

`KV_bytes_per_token = 2 * num_layers * num_kv_heads * head_dim * dtype_bytes`

The `2` is key and value. `num_kv_heads * head_dim` is the width of one of those vectors, summed over `num_layers` layers, at `dtype_bytes` per number. That is the entire formula. Read the four values from your model's `config.json` and compute it.

In [ ]:
# Read these four from your model's config.json (Qwen3-4B shown).
n_layers, n_kv_heads, head_dim, dtype_bytes = 36, 8, 128, 2
kv_per_tok = 2 * n_layers * n_kv_heads * head_dim * dtype_bytes
print(f"KV cache per token = 2 * {n_layers} * {n_kv_heads} * {head_dim} * {dtype_bytes} "
      f"= {kv_per_tok:,} bytes = {kv_per_tok/1024:.0f} KB")

# That number sets your concurrency. After weights, the card's free memory is cache.
card_gb, weights_gb, util = 20.0, 6.0, 0.70   # served FP8 4B, under-tuned default
kv_gb = util*card_gb - weights_gb
slots = kv_gb*1e9 / kv_per_tok
print(f"cache budget = {util}*{card_gb:.0f} - {weights_gb:.0f} = {kv_gb:.1f} GB")
print(f"  -> {slots:,.0f} token-slots, ~{slots/2048:.0f} concurrent 2048-token requests")

**What you should see:** about `144 KB` per token and room for roughly `26` full-length (2048-token) requests at the under-tuned `0.7` default. Module 9 raises utilization to `0.9` and that climbs to about `40`. That ceiling is what Module 8 drives into. Now confirm the server agrees: a longer generation should move the live cache gauge.

In [ ]:
# Measure: the cache fills DURING a request and frees after, so sample it in a
# background thread while several requests run, and catch the peak.
import threading
from concurrent.futures import ThreadPoolExecutor
client = build_client(settings)
peak = {"kv": 0.0}; stop = threading.Event()

def sample():
    while not stop.is_set():
        try: peak["kv"] = max(peak["kv"], metrics.snapshot(settings.metrics_url)["vllm:gpu_cache_usage_perc"])
        except Exception: pass
        time.sleep(0.05)

def long_gen(_):
    client.chat.completions.create(model=settings.model_name,
        messages=[{"role":"user","content":"Write a long, detailed essay about GPU memory and inference."}],
        max_tokens=512, temperature=0.0)

t = threading.Thread(target=sample, daemon=True); t.start()
with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(long_gen, range(8)))
stop.set(); t.join(timeout=1)
print(f"peak KV cache usage with 8 requests in flight: {peak['kv']*100:.1f}%")
print("Those are 8 requests' worth of keys and values, the bytes you sized, live on the card.")

**What you should see:** a non-zero peak, a few to several percent, that appears only while the requests are in flight and frees the moment they finish. Eight requests still barely dent the budget you computed, and that is the point: the cache is sized for many concurrent sequences, which is exactly what Module 3 drives into it.

## 6. GQA: the 4x you already have

Look again at the formula: the cache scales with `num_kv_heads`, not the number of query heads. Grouped-query attention shares one key/value head across several query heads, so a model with 32 query heads but 8 key/value heads pays for 8. That is a 4x cut in cache cost, for free, and it is why every modern model uses it.

In [ ]:
# Full multi-head attention would size the cache by the QUERY heads.
n_query_heads = 32
mha = 2 * n_layers * n_query_heads * head_dim * dtype_bytes
gqa = 2 * n_layers * n_kv_heads   * head_dim * dtype_bytes
print(f"full MHA ({n_query_heads} kv heads): {mha/1024:.0f} KB/token")
print(f"GQA      ({n_kv_heads} kv heads): {gqa/1024:.0f} KB/token   ({mha/gqa:.0f}x less)")
print(f"-> {n_query_heads//n_kv_heads}x more concurrent context for the same memory.")

See it in the cache shape itself. The cache axes are (batch, heads, sequence, head_dim). GQA shrinks the heads axis, so the whole cache shrinks with it.

In [ ]:
# Build the K cache tensor for a short sequence and read its shape. GQA shrinks the heads axis.
seq = 16
k_mha = rng.standard_normal((1, n_query_heads, seq, head_dim))   # full MHA caches every query head
k_gqa = rng.standard_normal((1, n_kv_heads,    seq, head_dim))   # GQA caches only the kv heads
print("MHA K-cache (batch, heads, seq, dim):", k_mha.shape, "->", int(k_mha.size * dtype_bytes / 1024), "KB")
print("GQA K-cache (batch, heads, seq, dim):", k_gqa.shape, "->", int(k_gqa.size * dtype_bytes / 1024), "KB")
print(f"GQA caches {n_query_heads}//{n_kv_heads} = {n_query_heads // n_kv_heads}x fewer heads, "
      f"so the cache is {n_query_heads // n_kv_heads}x smaller.")

**What you should see:** the GQA cache has 8 heads where full attention would have 32, so its shape and its KB are 4x smaller. That 4x is free, baked into the model architecture.

**What you should see:** `576 KB` versus `144 KB`, a 4x reduction. Open your own model's `config.json` and read `num_attention_heads` against `num_key_value_heads` to find your ratio.

## 7. Why decode is memory-bound: arithmetic intensity

Prefill and decode run the same weights, but one is fast and one is slow, and the reason is a single ratio: **arithmetic intensity**, the FLOPs you do per byte you read from memory.

- Prefill is a **GEMM** (matrix times matrix). For an `n x n` multiply it does `2n^3` FLOPs reading `3n^2` numbers, so intensity is `(2/3)n`. At `n = 4096` that is over `1300` FLOPs per byte. The GPU reads a weight once and reuses it across many tokens.
- Decode is a **GEMV** (matrix times one vector). It does about `1` FLOP per element it reads, roughly `0.5` FLOPs per byte at fp16. It reads a weight and uses it once.

Every GPU has a **crossover point**: `peak_FLOPS / bandwidth`, the intensity where it stops waiting on memory and starts waiting on math. Below it you are memory-bound. Decode sits far below it on every card made.

In [ ]:
# Derive both, and place them against your card's crossover point.
def gemm_intensity(n): return (2*n**3) / (3*n**2 * dtype_bytes)   # FLOPs per byte
def gemv_intensity(M, K): return (2*M*K) / ((M*K + K + M) * dtype_bytes)

n = 4096
print(f"prefill GEMM (n={n}) : {gemm_intensity(n):.0f} FLOPs/byte  (compute-bound)")
print(f"decode  GEMV (4096x4096): {gemv_intensity(4096,4096):.1f} FLOPs/byte  (memory-bound)")

peak_tflops, bw_tbs = 107.0, 0.36      # your card's datasheet
crossover = peak_tflops*1e12 / (bw_tbs*1e12)
print(f"\nyour card crossover point = {peak_tflops} TFLOP/s / {bw_tbs} TB/s = {crossover:.0f} FLOPs/byte")
print(f"decode uses ~{gemv_intensity(4096,4096)/crossover*100:.1f}% of the compute the card can do.")

**What you should see:** prefill above 1000 FLOPs/byte, decode below 1, and a crossover point in the hundreds. Decode runs at a fraction of a percent of the card's compute. The tensor cores are idle, waiting for weights to arrive from memory. That is not a tuning problem, it is the shape of the operation.

## 8. Measure it: recover your GPU's bandwidth from the token rate

If decode is memory-bound, then the time to make one token is the time to stream every weight out of memory once: `TPOT ~= weight_bytes / bandwidth`. Turn it around. Measure how fast the server decodes, and you have measured your card's effective memory bandwidth, with no profiler.

Stream a generation, time the gaps between tokens, and invert.

In [ ]:
# Stream, measure the inter-token gap (TPOT), and back out bandwidth.
start = time.time(); first = None; stamps = []
stream = client.chat.completions.create(model=settings.model_name,
    messages=[{"role":"user","content":"List the numbers 1 through 120, one per line."}],
    max_tokens=200, temperature=0.0, stream=True)
for chunk in stream:
    if not chunk.choices[0].delta.content: continue
    now = time.time()
    if first is None: first = now
    stamps.append(now)
gaps = [b-a for a,b in zip(stamps, stamps[1:])]
tpot = sum(gaps)/len(gaps)
weights_bytes = 6e9                            # served FP8-dynamic 4B, about 6 GB
bw = weights_bytes / tpot / 1e9               # GB/s

print(f"measured TPOT      : {tpot*1000:.1f} ms/token  ({1/tpot:.0f} tokens/s)")
print(f"recovered bandwidth: ~{bw:.0f} GB/s   (weight bytes / TPOT)")
print("Compare to your card's datasheet. It is a lower bound, because the server")
print("also does attention and overhead, but it proves decode is bandwidth-bound.")

**What you should see:** a TPOT of a few to tens of milliseconds, and a recovered bandwidth in the same ballpark as your card's spec (a bit lower, since the server is not a pure memory benchmark). You just measured a hardware property of your GPU from the rate it speaks. That only works because decode is memory-bound.

## 9. Batching lifts decode off the floor

One request reads every weight to make one token: intensity about 1. Run `B` requests together and the server reads each weight once and uses it for `B` tokens, so intensity scales with `B`. You walk decode up toward the crossover point. This is the whole reason a server batches, and you can watch it: aggregate token throughput should climb with concurrency even though each request barely changes.

In [ ]:
# Fire B requests concurrently, measure aggregate decode throughput.
from concurrent.futures import ThreadPoolExecutor
def one():
    r = client.chat.completions.create(model=settings.model_name,
        messages=[{"role":"user","content":"Write one paragraph about Kubernetes."}],
        max_tokens=128, temperature=0.0)
    return r.usage.completion_tokens

print(f"{'batch':>6} {'tokens/s':>10}  (one weight load now serves 'batch' tokens)")
for B in [1, 4, 16, 64]:
    t = time.time()
    with ThreadPoolExecutor(max_workers=B) as pool:
        toks = sum(pool.map(lambda _: one(), range(B)))
    print(f"{B:>6} {toks/(time.time()-t):>10.0f}")

**What you should see:** throughput rising sharply with batch size, several times the batch-1 rate, because the same weight read now feeds many tokens. That is decode climbing from memory-bound toward compute-bound. Module 8 watches this batch form in the metrics and drives it to saturation.

## 10. Reuse the cache across requests: prefix caching

You built the KV cache by hand and watched decode reuse it within one request. An agent reuses it across requests. Every step of a Thought, Action, Observation loop resends the same system prompt and tool schemas, commonly 2,000 to 3,000 tokens, and only the newest turn is new. vLLM's automatic prefix caching caches that shared prefix, so step 2 and beyond skip the prefill for it and jump straight to decode. This is the single biggest inference win for an agent on a GPU you own.

It is on by default in the vLLM V1 engine, so you prove it here, you do not enable it. Send the same long agent prompt twice and watch the first token go from a few hundred milliseconds to almost nothing while the hit counter climbs.

In [ ]:
# A streaming call that also reads the prefix-cache counters from /metrics before and after.
import uuid
from common import agent_loop, ttft_metrics
from common.config import build_client
client = build_client(settings)

def _prefix_counts():
    p = ttft_metrics._scrape()
    return (ttft_metrics._first(p, ttft_metrics._COUNTER["prefix_queries"]) or 0,
            ttft_metrics._first(p, ttft_metrics._COUNTER["prefix_hits"]) or 0)

def call(system, question, max_tokens=8):
    q0, h0 = _prefix_counts()
    r = agent_loop.step(client, settings.model_name,
        [{"role": "system", "content": system}, {"role": "user", "content": question}],
        max_tokens=max_tokens)
    q1, h1 = _prefix_counts()
    r["queried"] = int(q1 - q0); r["cached_hits"] = int(h1 - h0)
    return r

# A ~1,700-token agent prefix (system prompt + ~50 tool schemas). The RUN_ID nonce at the very
# FRONT forces a cold run each time you re-run the notebook, and it doubles as the block-0 lesson:
# change one token at the front and all reuse downstream is lost.
RUN_ID = uuid.uuid4().hex
base, approx = agent_loop.build_agent_prompt(n_tools=50)
SYSTEM = f"[build {RUN_ID}]\n" + base
print("prefix tokens (approx):", approx, "  (keep under ~1900 so the turn fits the 2048 cap)")

Send the same prefix twice. The only thing that differs is the short user turn, exactly like a new Observation in an agent loop.

In [ ]:
# Requires a live vLLM endpoint.
cold = call(SYSTEM, "Step 1: outline a plan to compare two GPUs.")
warm = call(SYSTEM, "Step 2: now refine the plan with cost in mind.")
print("COLD:", {k: cold[k] for k in ("ttft_ms", "queried", "cached_hits")})
print("WARM:", {k: warm[k] for k in ("ttft_ms", "queried", "cached_hits")})
print("warm hit-rate: %.0f%%" % (100 * warm["cached_hits"] / max(warm["queried"], 1)))
print("TTFT speedup: %.1fx" % (cold["ttft_ms"] / max(warm["ttft_ms"], 0.1)))

**What you should see:** the cold call prefills the whole ~1,700-token prefix, with cached hits near zero. The warm call reuses it: cached hits jump to nearly the whole prefix and TTFT collapses, often to a few percent of the cold time. That collapse is your agent's resent system prompt becoming free.

In [ ]:
# One picture: TTFT cold vs warm, and the hit counter climbing.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3.2))
a.bar(["cold (1st call)", "warm (2nd call)"], [cold["ttft_ms"], warm["ttft_ms"]],
      color=["#c0392b", "#27ae60"])
a.set_ylabel("TTFT (ms)"); a.set_title("Prefill collapses on the 2nd call")
b.bar(["queried", "cached hits"], [warm["queried"], warm["cached_hits"]],
      color=["#7f8c8d", "#2980b9"])
b.set_title("prefix cache hits climb"); fig.tight_layout()

Now run it as an agent does: five steps, same prefix, a new observation each time. Only step 1 pays the full prefill.

In [ ]:
# Requires a live vLLM endpoint. Every ReAct step reuses the prefix; only step 1 is tall.
steps = [call(SYSTEM, f"Observation {i}: the tool returned 3 hits. Next thought?") for i in range(1, 6)]
plt.figure(figsize=(7, 3))
plt.plot(range(1, 6), [s["ttft_ms"] for s in steps], "o-")
plt.xlabel("agent step"); plt.ylabel("TTFT (ms)")
plt.title("TTFT per ReAct step with prefix caching"); plt.tight_layout()
print("cached hits per step:", [s["cached_hits"] for s in steps])

**What you should see:** step 1 is a tall bar (the prefix is prefilled once), and steps 2 to 5 are flat near the floor (the prefix is reused). This is every step of your agent's loop. The system prompt and tool schemas are prefilled once and reused for free.

**Try the block-0 twist:** change one character inside `RUN_ID` before you call again, and the cached hits drop back to zero. One token at the front invalidates everything after it, which is why volatile fields (timestamps, session ids, a hash of the latest tool output) go at the tail, never the front. To see the no-cache baseline for real, you would start a second vLLM with `--no-enable-prefix-caching`.

## Things to know

- **The KV cache is the trade at the heart of inference.** You exchange `O(n^2)` recompute for `O(n)` memory. Everything downstream, paging, quantization, batching limits, is managing that memory.
- **Prefill and decode are different machines.** Prefill is a GEMM and compute-bound, so it sets time to first token. Decode is a GEMV and memory-bound, so it sets time per output token. They want opposite optimizations.
- **Decode speed is a memory property, not a compute one.** Until you batch, the tensor cores sit idle. The card's bandwidth, not its FLOPS, sets your single-request token rate.
- **GQA is free concurrency.** The cache scales with key/value heads. Fewer of them, more requests fit. Read both head counts from `config.json`.
- **PagedAttention is the next layer.** A naive cache reserves the full context length per request and wastes most of it. vLLM stores the cache in 16-token blocks and hands them out on demand, which is what lets the gauge in section 5 fill efficiently. The [PagedAttention paper](https://arxiv.org/abs/2309.06180) is the reference.
- **Your context is 2048 tokens by default.** `--max-model-len=2048` caps prompt plus generation, so a full-length request is 2048 tokens. To fill the cache you raise concurrency, not prompt length, until Module 9 raises the cap.

## Try it yourself

**Push the naive loop until it hurts.** Raise `ns` in section 3 to include 192 and 256 and re-run both. The naive time should roughly quadruple as you double `n`, while the cached time roughly doubles. Read the ratio off the plot.

**Change the model in the memory math.** In section 5, swap in the `num_layers`, `num_kv_heads`, and `head_dim` of a model you care about (an 8B, a 70B). Watch the per-token cost and the concurrent-request ceiling move. This is the calculation you do before you pick a card.

**Find your real crossover point.** In section 7, put your card's actual peak fp16 TFLOPS and bandwidth in, then run section 8 and mark which batch sizes are still left of the crossover. Those are the ones where adding concurrency is still free throughput.

## Summary

- You implemented attention, and saw prefill is a `p x p` GEMM over the whole prompt at once.
- You wrote naive generation and measured its `np + n(n-1)/2` cost, then added a KV cache and measured the drop to `O(n)`. That trade, compute for memory, is the foundation.
- You derived `2 * layers * kv_heads * head_dim * bytes` per token, computed it for your model, and watched the live cache move.
- You separated prefill (GEMM, compute-bound) from decode (GEMV, memory-bound) by arithmetic intensity, recovered your card's bandwidth from the decode rate, and watched batching lift decode toward the compute roof.

## Next

**Module 4: Dense versus MoE on the GPU.** You proved decode is memory-bound. Next you draw that as a plot, the compute and memory limits meeting at a crossover, and use it to explain why a Mixture-of-Experts model generates faster than its size suggests.